# Linear Algebra and Multivariate Linear Regression

**CS156 - Finding Patterns in Data with Machine Learning**

This notebook organizes the Session 3 PCW around multivariate ordinary least squares (OLS):

- representing several predictors with matrices,
- predicting Titanic passenger fare from `Pclass`, `Age`, and `SibSp`,
- writing the OLS loss in vector form,
- differentiating the loss and deriving the normal equations,
- solving a three-point system exactly,
- interpreting residuals geometrically,
- calculating the sum of squared residuals on the full dataset.

The goal is to connect the matrix notation directly to the code.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression

## 1. Linear algebra view of regression

A matrix can be thought of as a collection of observations in a vector space.

With three predictors, each passenger can be represented as

$$
x_i =
\begin{bmatrix}
\text{Pclass}_i \\
\text{Age}_i \\
\text{SibSp}_i
\end{bmatrix}.
$$

Stacking the passenger vectors into rows gives the design matrix $X$.

A multivariate linear model predicts fare with

$$
\hat y = X\beta,
$$

or, when an intercept is included,

$$
\hat y = \beta_0\mathbf{1}+X\beta.
$$

Geometrically, the model describes a hyperplane in the predictor space.

## 2. Load the Titanic data

The original PCW used Kaggle's Titanic `train.csv`. For a reproducible notebook, the cell below uses a public mirror of the same training dataset and keeps only the columns used in the PCW.

Rows with missing `Age` values are removed before fitting.

In [ ]:
TITANIC_URL = (
    "https://raw.githubusercontent.com/"
    "datasciencedojo/datasets/master/titanic.csv"
)

titanic = pd.read_csv(TITANIC_URL)

df = titanic[["Fare", "Pclass", "Age", "SibSp"]].dropna().copy()

X = df[["Pclass", "Age", "SibSp"]]
y = df["Fare"]

df.head()

### Fit the full multivariate model

Scikit-learn includes an intercept by default. The fitted model therefore has four learned coefficients in total: one intercept plus one coefficient for each of the three predictors.

In [ ]:
model = LinearRegression()
model.fit(X, y)

print("Intercept:", model.intercept_)

for feature, coefficient in zip(X.columns, model.coef_):
    print(f"{feature}: {coefficient}")

## 3. Ordinary least squares in matrix notation

Let

$$
\hat y=X\beta
$$

and define the residual vector

$$
r=y-X\beta.
$$

The sum of squared residuals is

$$
L(\beta)
=
\|y-X\beta\|_2^2
=
(y-X\beta)^T(y-X\beta).
$$

Expanding,

$$
L(\beta)
=
y^Ty
-
2\beta^TX^Ty
+
\beta^TX^TX\beta.
$$

Taking the gradient with respect to $\beta$ gives

$$
\nabla_\beta L
=
-2X^Ty
+
2X^TX\beta.
$$

At the least-squares solution,

$$
\nabla_\beta L=0,
$$

so

$$
X^TX\hat\beta=X^Ty.
$$

If $X^TX$ is invertible,

$$
\boxed{
\hat\beta=(X^TX)^{-1}X^Ty
}
$$

These are the **normal equations**.

### Spatial interpretation

The columns of $X$ span a subspace of $\mathbb{R}^n$ containing every prediction vector the model can produce.

OLS selects the prediction

$$
\hat y=X\hat\beta
$$

that is closest to the observed target vector $y$ in Euclidean distance.

At the optimum,

$$
X^T(y-X\hat\beta)=0.
$$

Therefore the residual vector is orthogonal to every column of $X$. In other words, OLS projects $y$ onto the column space of $X$.

## 4. Three passengers: a square 3 × 3 system

The PCW selected three Titanic observations:

| Passenger | Pclass | Age | SibSp | Fare |
|---|---:|---:|---:|---:|
| 1 | 3 | 22 | 1 | 7.2500 |
| 2 | 1 | 38 | 1 | 71.2833 |
| 3 | 3 | 26 | 0 | 7.9250 |

For this exercise we use exactly three predictors **without a separate intercept**, producing a 3 × 3 system.

Because there are three equations and three parameters, an invertible $X$ can fit these three points exactly. That exact interpolation does **not** imply that the resulting model will generalize well.

In [ ]:
X_three = np.array(
    [
        [3.0, 22.0, 1.0],
        [1.0, 38.0, 1.0],
        [3.0, 26.0, 0.0],
    ]
)

y_three = np.array([7.25, 71.2833, 7.925])

beta_three = np.linalg.solve(X_three, y_three)
pred_three = X_three @ beta_three
residuals_three = y_three - pred_three
ssr_three = residuals_three @ residuals_three

print("beta =", beta_three)
print("predictions =", pred_three)
print("residuals =", residuals_three)
print("SSR =", ssr_three)

The three fitted coefficients correspond to:

$$
\text{Fare}
=
\beta_1(\text{Pclass})
+
\beta_2(\text{Age})
+
\beta_3(\text{SibSp}).
$$

The PCW obtained approximately:

- $\beta_1=-15.3807$ for `Pclass`,
- $\beta_2=2.0795$ for `Age`,
- $\beta_3=7.6430$ for `SibSp`.

Because this square system is exactly determined, its SSR is numerically zero up to floating-point error.

## 5. Visualizing one slice of the multivariate model

A three-variable linear model cannot be represented completely by a single 2D line.

One useful visualization is to vary `Age` while holding `Pclass` and `SibSp` fixed at their sample means. The resulting line is a **slice** through the higher-dimensional fitted surface.

In [ ]:
pclass = X_three[:, 0]
age = X_three[:, 1]
sibsp = X_three[:, 2]

pclass_mean = pclass.mean()
sibsp_mean = sibsp.mean()

age_grid = np.linspace(age.min() - 2, age.max() + 2, 100)

fare_line = (
    beta_three[0] * pclass_mean
    + beta_three[1] * age_grid
    + beta_three[2] * sibsp_mean
)

plt.figure(figsize=(8, 6))
plt.scatter(age, y_three, s=100, label="Observed data points")
plt.plot(age_grid, fare_line, linewidth=2, label="Age slice of fitted model")

for i, (x_i, y_i) in enumerate(zip(age, y_three), start=1):
    plt.annotate(f"P{i}", (x_i, y_i), xytext=(6, 6), textcoords="offset points")

plt.xlabel("Age")
plt.ylabel("Fare")
plt.title("Fare vs. age with Pclass and SibSp held constant")
plt.legend()
plt.grid(True)
plt.show()

## 6. Sum of squared residuals on the full dataset

For the full model,

$$
SSR
=
\sum_i(y_i-\hat y_i)^2
=
r^Tr.
$$

Spatially, $\sqrt{SSR}=\|r\|_2$ is the Euclidean distance from $y$ to its projection $\hat y$ in observation space. Squaring that distance gives SSR.

In [ ]:
y_pred = model.predict(X)

residuals = y.to_numpy() - y_pred
ssr = residuals.T @ residuals

print("SSR:", ssr)

## 7. Why square the residuals?

Squaring has several useful consequences:

- positive and negative residuals do not cancel;
- large errors are penalized strongly;
- the objective is smooth and differentiable;
- the geometry becomes Euclidean distance;
- the derivative leads to a tractable linear system.

Absolute residuals are also meaningful, but they lead to a different optimization problem.

## 8. Main takeaways

1. A design matrix stores observations as rows and predictors as columns.
2. Multivariate regression learns a hyperplane rather than a single 2D line.
3. OLS minimizes the squared length of the residual vector.
4. The normal equations arise by differentiating the matrix-form loss.
5. The least-squares prediction is an orthogonal projection onto the column space of the design matrix.
6. Three parameters can interpolate three linearly independent observations exactly, but that says nothing by itself about generalization.